In [2]:
from jaad_data import JAAD

jaad_api = JAAD(data_path = '.')

In [3]:
# Extract & Save Images
# JAAD has in-built method extract_and_save_images, but it extracts images in png format, which takes up too much disk space.
import os
import cv2

def extract_all_images(jaad_obj, image_ext=".jpg", overwrite=False):
    clip_files = sorted(
        f for f in os.listdir(jaad_obj._clips_path)
        if f.lower().endswith(".mp4")
    )

    os.makedirs(jaad_obj._images_path, exist_ok=True)
    print(f"Found {len(clip_files)} clips in {jaad_obj._clips_path}")

    for i, clip_file in enumerate(clip_files, start=1):
        vid = os.path.splitext(clip_file)[0]
        clip_path = os.path.join(jaad_obj._clips_path, clip_file)
        save_dir = os.path.join(jaad_obj._images_path, vid)
        os.makedirs(save_dir, exist_ok=True)

        cap = cv2.VideoCapture(clip_path)
        if not cap.isOpened():
            print(f"[{i}/{len(clip_files)}] [SKIP] cannot open: {clip_path}")
            continue

        frame_idx = 0
        saved = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            out_path = os.path.join(save_dir, f"{frame_idx:05d}{image_ext}")
            if overwrite or not os.path.exists(out_path):
                cv2.imwrite(out_path, frame)
                saved += 1
            frame_idx += 1

        cap.release()
        print(f"[{i}/{len(clip_files)}] {vid}: saved {saved} / {frame_idx} frames")

# Run extraction for all videos
extract_all_images(jaad_api, image_ext=".jpg", overwrite=False)

Found 346 clips in .\JAAD_clips
[1/346] video_0001: saved 0 / 600 frames
[2/346] video_0002: saved 0 / 210 frames
[3/346] video_0003: saved 0 / 210 frames
[4/346] video_0004: saved 0 / 180 frames
[5/346] video_0005: saved 0 / 240 frames
[6/346] video_0006: saved 0 / 330 frames
[7/346] video_0007: saved 0 / 120 frames
[8/346] video_0008: saved 0 / 150 frames
[9/346] video_0009: saved 0 / 120 frames
[10/346] video_0010: saved 0 / 90 frames
[11/346] video_0011: saved 0 / 270 frames
[12/346] video_0012: saved 0 / 120 frames
[13/346] video_0013: saved 0 / 150 frames
[14/346] video_0014: saved 0 / 270 frames
[15/346] video_0015: saved 0 / 420 frames
[16/346] video_0016: saved 0 / 210 frames
[17/346] video_0017: saved 0 / 270 frames
[18/346] video_0018: saved 0 / 390 frames
[19/346] video_0019: saved 0 / 480 frames
[20/346] video_0020: saved 0 / 540 frames
[21/346] video_0021: saved 0 / 180 frames
[22/346] video_0022: saved 0 / 420 frames
[23/346] video_0023: saved 0 / 240 frames
[24/346] vid

In [4]:
# Generate Database
db = jaad_api.generate_database()

---------------------------------------------------------
Generating database for jaad
jaad database loaded from c:\Users\tengc\OneDrive - National University of Singapore\NUS modules\Y3S2 modules\CS3264\Projects\CS3264 Source Files\CS3264_Team6_Project\data_cache\jaad_database.pkl


In [5]:
# Extract Features from Pedestrians
# Notes on meaning of Behavior Annotations:
    # occlusion: 0(not occluded), 1(partially occluded), 2(fully occluded)
    # cross: 0(not crossing), 1(crossing)
    # reaction: 0(no reaction), 1(reaction)
    # hand_gesture: 0(no hand gesture), 1(hand gesture)
    # look: 0(not looking), 1(looking)
    # action: 0(Standing), 1(Walking)
    # nod: 0(no nod), 1(nod)
# Notes on meaning of Pedestrian Attributes:
    # old_id: Original Annotation ID String
    # age: 0(child), 1 (young), 2 (adult), 3 (senior)
    # crossing: 0(not crossing), 1(crossing), -1(irrelevant)
    # crossing_point: Frame Index of Crossing Point (if crossing), -1 otherwise
    # decision_point: Frame Index of Decision Point (if crossing), -1 otherwise
    # designated: 0(not designated crossing point), 1(designated crossing point)
    # gender: 0(n/a), 1(female), 2(male)
    # group_size: Number of People in Group
    # intersection: 0(not at intersection), 1(at intersection)
    # motion_direction: 0(n/a), 1(Lateral / Across), 2(Longitudinal / Along)
    # num_lanes: Number of Road Lanes
    # signalized: 0(n/a), 1(non-signalized intersection), 2(signalized intersection)
    # traffic_direction: 0(One-Way), 1(Two-Way)

pedestrian_ids = jaad_api._get_pedestrian_ids()

features = []
for vid, video in db.items():
    for pid, pedestrian in video["ped_annotations"].items():
        if 'b' not in pid: # Skip Pedestrians without Behavior Annotations
            continue
        
        # Dynamic Features (Time-Series)
        frames = pedestrian.get("frames", [])
        bboxes = pedestrian.get("bbox", [])
        occlusions = pedestrian.get("occlusion", [])
        behavior = pedestrian.get("behavior", {})

        # Static Features (Non-Time-Series)
        attributes = pedestrian.get("attributes", {})

        for i, frame in enumerate(frames):
            row = {
                "video_id": vid,
                "pedestrian_id": pid,

                # Frames
                "frame_id": frame,

                # BBoxes
                "bbox_x1": bboxes[i][0],
                "bbox_y1": bboxes[i][1],
                "bbox_x2": bboxes[i][2],
                "bbox_y2": bboxes[i][3],

                # Occlusions
                "occlusion": occlusions[i],

                # Behavior Annotations
                "cross": behavior.get("cross", [0]*len(frames))[i],
                "reaction": behavior.get("reaction", [0]*len(frames))[i],
                "hand_gesture": behavior.get("hand_gesture", [0]*len(frames))[i],
                "look": behavior.get("look", [0]*len(frames))[i],
                "action": behavior.get("action", [0]*len(frames))[i],
                "nod": behavior.get("nod", [0]*len(frames))[i],

                # Pedestrian Attributes
                "old_id": attributes.get("old_id", ""),
                "age": attributes.get("age", 0),
                "crossing": attributes.get("crossing", 0),
                "crossing_point": attributes.get("crossing_point", 0),
                "decision_point": attributes.get("decision_point", 0),
                "designated": attributes.get("designated", 0),
                "gender": attributes.get("gender", 0),
                "group_size": attributes.get("group_size", 1),
                "intersection": attributes.get("intersection", 0),
                "motion_direction": attributes.get("motion_direction", 0),
                "num_lanes": attributes.get("num_lanes", 0),
                "signalized": attributes.get("signalized", 0),
                "traffic_direction": attributes.get("traffic_direction", 0)
            }
            features.append(row)


import pandas as pd

df = pd.DataFrame(features)
print(f"Features shape: {df.shape}")
display(df.head())

# Count number of unique pedestrians
unique_pedestrians = df["pedestrian_id"].nunique()
assert unique_pedestrians == len([pid for pid in pedestrian_ids if 'b' in pid]), "Mismatch between unique pedestrians in features and pedestrian IDs."

# Check for Class Imbalance in Crossing Behavior
crossing_counts = df["crossing"].value_counts()
display(crossing_counts)

---------------------------------------------------------
Generating database for jaad
jaad database loaded from c:\Users\tengc\OneDrive - National University of Singapore\NUS modules\Y3S2 modules\CS3264\Projects\CS3264 Source Files\CS3264_Team6_Project\data_cache\jaad_database.pkl
Features shape: (132700, 27)


,video_id,pedestrian_id,frame_id,bbox_x1,bbox_y1,bbox_x2,bbox_y2,occlusion,cross,reaction,...,crossing_point,decision_point,designated,gender,group_size,intersection,motion_direction,num_lanes,signalized,traffic_direction
0,video_0001,0_1_3b,0,465.0,730.0,533.0,848.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1
1,video_0001,0_1_3b,1,463.0,730.0,532.0,848.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1
2,video_0001,0_1_3b,2,461.0,730.0,531.0,849.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1
3,video_0001,0_1_3b,3,459.0,730.0,530.0,849.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1
4,video_0001,0_1_3b,4,458.0,731.0,530.0,851.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1


crossing
 1    105026
-1     16216
 0     11458
Name: count, dtype: int64

In [6]:
# Train XGBoost classifier to predict crossing behavior (frame-level binary classification)

import os
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve, classification_report, roc_auc_score
from xgboost import XGBClassifier

# Safety-focused threshold shared by evaluation and video rendering
DECISION_THRESHOLD = 0.25

# Keep names consistent for training/inference/video rendering
cat_cols = [
    "occlusion", "reaction", "hand_gesture", "look", "action", "nod",
    "age", "designated", "gender", "intersection",
    "motion_direction", "signalized", "traffic_direction"
]
num_cols = [
    "bbox_center_x", "bbox_center_y", "bbox_width", "bbox_height", "bbox_area",
    "velocity_x", "velocity_y", "speed", "acceleration_x", "acceleration_y",
    "group_size", "num_lanes"
]


def engineer_features(frame_df):
    """Compute derived geometry and motion features used by the model."""
    out = frame_df.copy()

    # JAAD frames are 1920 x 1080
    W, H = 1920.0, 1080.0

    # Normalized bbox geometry
    out["bbox_center_x"] = ((out["bbox_x1"] + out["bbox_x2"]) / 2.0) / W
    out["bbox_center_y"] = ((out["bbox_y1"] + out["bbox_y2"]) / 2.0) / H
    out["bbox_width"] = (out["bbox_x2"] - out["bbox_x1"]) / W
    out["bbox_height"] = (out["bbox_y2"] - out["bbox_y1"]) / H
    out["bbox_area"] = out["bbox_width"] * out["bbox_height"]

    # Pedestrian velocity and acceleration (frame-to-frame differences)
    group_keys = ["video_id", "pedestrian_id"]
    out["velocity_x"] = out.groupby(group_keys)["bbox_center_x"].diff().fillna(0.0)
    out["velocity_y"] = out.groupby(group_keys)["bbox_center_y"].diff().fillna(0.0)
    out["speed"] = np.sqrt(out["velocity_x"] ** 2 + out["velocity_y"] ** 2)
    out["acceleration_x"] = out.groupby(group_keys)["velocity_x"].diff().fillna(0.0)
    out["acceleration_y"] = out.groupby(group_keys)["velocity_y"].diff().fillna(0.0)

    return out


def read_ids(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


# ----- DATA PREP -----
df = pd.DataFrame(features).copy()
df = df.sort_values(["video_id", "pedestrian_id", "frame_id"]).reset_index(drop=True)
df = engineer_features(df)

# Frame-level binary target only
df = df[df["cross"].isin([0, 1])].copy()

# ----- DATA SPLIT -----
split_root = os.path.join("split_ids", "default")
train_videos = read_ids(os.path.join(split_root, "train.txt"))
val_videos = read_ids(os.path.join(split_root, "val.txt"))
test_videos = read_ids(os.path.join(split_root, "test.txt"))
assert set(train_videos).isdisjoint(set(val_videos)), "Train and validation video IDs overlap."
assert set(train_videos).isdisjoint(set(test_videos)), "Train and test video IDs overlap."
assert set(val_videos).isdisjoint(set(test_videos)), "Validation and test video IDs overlap."

train_df = df[df["video_id"].isin(train_videos)].copy()
val_df = df[df["video_id"].isin(val_videos)].copy()
test_df = df[df["video_id"].isin(test_videos)].copy()
assert len(set(train_df.index).intersection(val_df.index)) == 0, "Train and validation frame indices overlap."
assert len(set(train_df.index).intersection(test_df.index)) == 0, "Train and test frame indices overlap."
assert len(set(val_df.index).intersection(test_df.index)) == 0, "Validation and test frame indices overlap."

print("Frame counts by split:")
print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})

X_train = train_df[cat_cols + num_cols]
y_train = train_df["cross"]
X_val = val_df[cat_cols + num_cols]
y_val = val_df["cross"]
X_test = test_df[cat_cols + num_cols]
y_test = test_df["cross"]

# Handle imbalance using training split only
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = float(neg) / float(pos) if pos > 0 else 1.0
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

# ----- MODEL TRAINING -----
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

clf = Pipeline([
    ("pre", pre),
    ("model", XGBClassifier(
        n_estimators=2500,
        max_depth=8,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
    ))
])

clf.fit(X_train, y_train)

# ----- EVALUATION -----
# Validation metrics
val_proba = clf.predict_proba(X_val)[:, 1]

val_pred = (val_proba >= DECISION_THRESHOLD).astype(int)
print(f"\nValidation threshold: {DECISION_THRESHOLD}")
print("Validation AUROC:", roc_auc_score(y_val, val_proba))


print(classification_report(y_val, val_pred))

# Test metrics
test_proba = clf.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= DECISION_THRESHOLD).astype(int)
print("Test AUROC:", roc_auc_score(y_test, test_proba))
print(classification_report(y_test, test_pred))

Frame counts by split:
{'train': 61805, 'val': 9583, 'test': 52966}
scale_pos_weight: 0.722

Validation threshold: 0.25
Validation AUROC: 0.9590220462295824
              precision    recall  f1-score   support

           0       0.94      0.81      0.87      4325
           1       0.86      0.96      0.91      5258

    accuracy                           0.89      9583
   macro avg       0.90      0.89      0.89      9583
weighted avg       0.90      0.89      0.89      9583

Test AUROC: 0.9054790636836672
              precision    recall  f1-score   support

           0       0.90      0.70      0.79     23243
           1       0.80      0.94      0.86     29723

    accuracy                           0.83     52966
   macro avg       0.85      0.82      0.83     52966
weighted avg       0.84      0.83      0.83     52966



In [7]:
# Visualization of predictions on test video frames for XGBoost classifier
import pandas as pd
import cv2
import os

def predict_from_df(df, clf, cat_cols, num_cols, threshold, video_id=None, pedestrian_id=None):
    """Return per-frame crossing probabilities directly from the feature DataFrame."""
    data = df.copy()

    if video_id is not None:
        data = data[data["video_id"] == video_id].copy()
    if pedestrian_id is not None:
        data = data[data["pedestrian_id"] == pedestrian_id].copy()

    if data.empty:
        raise ValueError("No rows matched the requested video_id / pedestrian_id filter.")

    data = data.sort_values(["video_id", "pedestrian_id", "frame_id"]).reset_index(drop=True)

    # Reuse the same feature logic used during model training
    if "bbox_center_x" not in data.columns:
        data = engineer_features(data)

    X = data.reindex(columns=cat_cols + num_cols, fill_value=0)
    scores = clf.predict_proba(X)[:, 1]
    result = data[[
        "video_id", "pedestrian_id", "frame_id",
        "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2",
        "cross"
    ]].copy()
    result["crossing_score"] = scores
    result["predicted_cross"] = (scores >= threshold).astype(int)
    return result


def write_prediction_video_from_df(df, obj, clf, cat_cols, num_cols, threshold, video_id, output_path, fps=30):
    """Write a video with bounding boxes, model prediction, and ground truth per frame."""
    predictions = predict_from_df(
        df, clf, cat_cols, num_cols, threshold=threshold, video_id=video_id
    )

    images_root = getattr(obj, "_images_path", os.path.join(".", "images"))
    video_path = os.path.join(images_root, video_id)
    if not os.path.isdir(video_path):
        raise FileNotFoundError(f"Video image folder not found: {video_path}")

    frame_files = sorted([
        f for f in os.listdir(video_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])
    if not frame_files:
        raise RuntimeError(f"No frames found in {video_path}. Expected .jpg or .png files.")

    first_frame = cv2.imread(os.path.join(video_path, frame_files[0]))
    if first_frame is None:
        raise RuntimeError(f"Could not read first frame: {frame_files[0]}")

    height, width = first_frame.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    if not writer.isOpened():
        raise RuntimeError(f"Could not open video writer for: {output_path}")

    pred_by_frame = {}
    for _, row in predictions.iterrows():
        pred_by_frame.setdefault(int(row["frame_id"]), []).append(row)

    for frame_file in frame_files:
        frame_idx = int(os.path.splitext(frame_file)[0])
        img = cv2.imread(os.path.join(video_path, frame_file))
        if img is None:
            continue

        for row in pred_by_frame.get(frame_idx, []):
            bbox = [row["bbox_x1"], row["bbox_y1"], row["bbox_x2"], row["bbox_y2"]]
            score = float(row["crossing_score"])
            pred_label = int(row["predicted_cross"])
            gt_label = int(row["cross"])
            color = (0, int(255 * (1 - score)), int(255 * score))
            cv2.rectangle(img, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), color, 3)
            label = f"{row['pedestrian_id']} Pred:{pred_label} GT:{gt_label} Score:{score:.2f}"
            cv2.putText(
                img, label, (int(bbox[0]), max(20, int(bbox[1] - 10))),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
            )

        writer.write(img)

    writer.release()
    return output_path


# Build videos for TEST split only using the same threshold as evaluation
test_ids_path = os.path.join("split_ids", "default", "test.txt")
test_video_ids = read_ids(test_ids_path)
output_dir = os.path.join(".", "predictions_test_videos")
os.makedirs(output_dir, exist_ok=True)

written = []
skipped = []
for video_id in test_video_ids:
    output_video_path = os.path.join(output_dir, f"{video_id}_predictions.mp4")
    try:
        write_prediction_video_from_df(
            df, jaad_api, clf, cat_cols, num_cols,
            threshold=DECISION_THRESHOLD,
            video_id=video_id,
            output_path=output_video_path
        )
        written.append(output_video_path)
        print(f"[OK] {video_id} -> {output_video_path}")
    except Exception as e:
        skipped.append((video_id, str(e)))
        print(f"[SKIP] {video_id}: {e}")

print(f"\nCreated {len(written)} test videos.")
print(f"Skipped {len(skipped)} test videos.")
if skipped:
    display(pd.DataFrame(skipped, columns=["video_id", "reason"]))

[OK] video_0005 -> .\predictions_test_videos\video_0005_predictions.mp4
[SKIP] video_0015: No rows matched the requested video_id / pedestrian_id filter.
[OK] video_0016 -> .\predictions_test_videos\video_0016_predictions.mp4
[OK] video_0017 -> .\predictions_test_videos\video_0017_predictions.mp4
[OK] video_0028 -> .\predictions_test_videos\video_0028_predictions.mp4
[SKIP] video_0036: No rows matched the requested video_id / pedestrian_id filter.
[OK] video_0042 -> .\predictions_test_videos\video_0042_predictions.mp4
[SKIP] video_0043: No rows matched the requested video_id / pedestrian_id filter.
[OK] video_0045 -> .\predictions_test_videos\video_0045_predictions.mp4
[OK] video_0046 -> .\predictions_test_videos\video_0046_predictions.mp4
[OK] video_0048 -> .\predictions_test_videos\video_0048_predictions.mp4
[OK] video_0053 -> .\predictions_test_videos\video_0053_predictions.mp4
[OK] video_0055 -> .\predictions_test_videos\video_0055_predictions.mp4
[SKIP] video_0058: No rows matched

,video_id,reason
0,video_0015,No rows matched the requested video_id / pedes...
1,video_0036,No rows matched the requested video_id / pedes...
2,video_0043,No rows matched the requested video_id / pedes...
3,video_0058,No rows matched the requested video_id / pedes...
4,video_0075,No rows matched the requested video_id / pedes...
5,video_0153,No rows matched the requested video_id / pedes...


In [8]:
# Assuming 'df' is your current dataframe containing: video_id, pedestrian_id, frame_id, and your model's predictions

# 1. Load the JAAD database annotations
jaad_db = jaad_api.generate_database()

# 2. Function to check if a pedestrian is crossing in a specific frame
def get_crossing_label(row):
    vid = row['video_id']
    ped = row['pedestrian_id']
    frame = row['frame_id']
    
    try:
        # Navigate the JAAD dict structure to find the pedestrian's behavioral data
        # Note: You may need to inspect jaad_db keys to match the exact path
        ped_data = jaad_db[vid]['pedestrians'][ped]
        
        # Check if 'crossing' behavior is active at this specific frame
        # JAAD usually stores a list of frames where the action occurs
        if frame in ped_data['behavior']['cross']:
            return 1
        else:
            return 0
    except KeyError:
        # Pedestrian or behavior data not found
        return 0

# 3. Apply the function to create the missing ground truth column
df['crossing_true'] = df.apply(get_crossing_label, axis=1)

# Now save it!
df.to_csv('baseline_predictions.csv', index=False)

---------------------------------------------------------
Generating database for jaad
jaad database loaded from c:\Users\tengc\OneDrive - National University of Singapore\NUS modules\Y3S2 modules\CS3264\Projects\CS3264 Source Files\CS3264_Team6_Project\data_cache\jaad_database.pkl


In [ ]:
import pandas as pd

print("Formatting Ground Truth and predicting XGBoost probabilities...")

# The ground truth is ALREADY perfectly extracted in the dataframe under the 'cross' column
# We just duplicate it to 'crossing_true' so the temporal notebook can read it
df['crossing_true'] = df['cross']

# Generate the missing baseline predictions using the trained XGBoost model
X_all = df.reindex(columns=cat_cols + num_cols, fill_value=0)
df['baseline_prediction'] = clf.predict_proba(X_all)[:, 1]

# Save this exact state for the temporal evaluation
output_cols = ['video_id', 'pedestrian_id', 'frame_id', 'bbox_y1', 'bbox_y2', 'crossing_true', 'baseline_prediction']
df[output_cols].to_csv('baseline_predictions.csv', index=False)

print("Saved baseline_predictions.csv with XGBoost probabilities and accurate Ground Truth for later pipelines")

Formatting Ground Truth and predicting XGBoost probabilities...
Saved baseline_predictions.csv with XGBoost probabilities and accurate Ground Truth!
